# Deploy Strands Decider 2B on SageMaker AI

Deploy the published [Strands Decider 2B checkpoint](https://huggingface.co/StrandsAgents/strands-decider-2B-hobson-v19)
on one `ml.g5.xlarge` real-time endpoint and invoke its Choice, Noul, and Score
primitives. This uses the official inference engine and preserves the LoRA,
pointer readout, and fitted temperatures.

Read [README.md](README.md) for prerequisites, model licenses, container
support limitations, and references. This is an evaluation example. The AWS
PyTorch 2.6 base image is beyond its published support period; upgrading
Torch at startup does not extend that support.

Run from this directory. Run the cleanup cell when finished, including after
a failed deployment once SageMaker allows endpoint deletion.

## 1. Environment preparation

In [ ]:
%pip install --upgrade 'boto3>=1.40,<2' --quiet

In [ ]:
import json
import math
import os
import statistics
import tarfile
import time
import uuid
from pathlib import Path

import boto3
from botocore.config import Config
from botocore.exceptions import ClientError

session = boto3.Session()
region = session.region_name or "us-east-1"
sm = session.client("sagemaker", region_name=region)
runtime = session.client(
    "sagemaker-runtime", region_name=region,
    config=Config(read_timeout=70, retries={"max_attempts": 0}),
)
s3 = session.client("s3", region_name=region)
iam = session.client("iam")
logs = session.client("logs", region_name=region)
identity = session.client("sts").get_caller_identity()
account_id = identity["Account"]

# Set this when the notebook's own role is not a SageMaker execution role.
execution_role = os.environ.get("SAGEMAKER_EXECUTION_ROLE_ARN")
bucket = os.environ.get("SAGEMAKER_ARTIFACT_BUCKET", f"sagemaker-{region}-{account_id}")
instance_type = "ml.g5.xlarge"
inference_image = os.environ.get(
    "SAGEMAKER_INFERENCE_IMAGE_URI",
    f"763104351884.dkr.ecr.{region}.amazonaws.com/"
    "pytorch-inference:2.6.0-gpu-py312-cu124-ubuntu22.04-sagemaker",
)
name = f"strands-decider-2b-{uuid.uuid4().hex[:12]}"
artifact_key = f"strands-decider/{name}/model.tar.gz"
artifact_version = None
example_dir = Path.cwd()
archive_path = example_dir / f"{name}.tar.gz"
for filename in ["inference.py", "requirements.txt", "example-request.json"]:
    if not (example_dir / filename).is_file():
        raise FileNotFoundError(f"Run the notebook from its example directory: missing {filename}")
print(f"Region: {region}; instance: {instance_type}; endpoint: {name}")

### Resolve the execution role

In Studio the current role may already be appropriate. Outside Studio,
set `SAGEMAKER_EXECUTION_ROLE_ARN` to an existing SageMaker role. This cell
reads the role's actual ARN through IAM, preserving paths such as
`service-role/`, and checks its SageMaker trust.

In [ ]:
if execution_role is None:
    caller_arn = identity["Arn"]
    if ":assumed-role/" not in caller_arn:
        raise ValueError("Set SAGEMAKER_EXECUTION_ROLE_ARN to an existing SageMaker execution role.")
    role_name = caller_arn.split(":assumed-role/", 1)[1].split("/", 1)[0]
else:
    role_name = execution_role.rsplit("/", 1)[-1]
role_info = iam.get_role(RoleName=role_name)["Role"]
if execution_role is not None and execution_role != role_info["Arn"]:
    raise ValueError("The configured execution-role ARN does not match the resolved IAM role.")
execution_role = role_info["Arn"]
trusts_sagemaker = any(
    statement.get("Effect") == "Allow"
    and "sagemaker.amazonaws.com" in (
        [statement.get("Principal", {}).get("Service")]
        if isinstance(statement.get("Principal", {}).get("Service"), str)
        else statement.get("Principal", {}).get("Service", [])
    )
    for statement in role_info["AssumeRolePolicyDocument"]["Statement"]
)
if not trusts_sagemaker:
    raise ValueError(
        "The current role does not trust SageMaker. Set SAGEMAKER_EXECUTION_ROLE_ARN "
        "to an appropriate existing execution role and rerun setup."
    )
print("Execution role resolved and SageMaker trust verified.")

## 2. Prepare the model artifact

Package the companion handler and requirements under `code/`, as expected
by the SageMaker PyTorch container. Requirements install the official runtime
at a pinned source commit. The handler downloads the pinned checkpoint and
base weights when the model worker starts.

The default artifact bucket is shared and is preserved during cleanup.

In [ ]:
try:
    s3.head_bucket(Bucket=bucket)
except ClientError as error:
    if error.response["Error"]["Code"] not in {"404", "NoSuchBucket", "NotFound"}:
        raise
    create_args = {"Bucket": bucket}
    if region != "us-east-1":
        create_args["CreateBucketConfiguration"] = {"LocationConstraint": region}
    s3.create_bucket(**create_args)
    s3.put_public_access_block(
        Bucket=bucket,
        PublicAccessBlockConfiguration={
            "BlockPublicAcls": True, "IgnorePublicAcls": True,
            "BlockPublicPolicy": True, "RestrictPublicBuckets": True,
        },
    )

with tarfile.open(archive_path, "w:gz") as archive:
    for filename in ["inference.py", "requirements.txt"]:
        archive.add(example_dir / filename, arcname=f"code/{filename}")
with archive_path.open("rb") as artifact:
    uploaded = s3.put_object(Bucket=bucket, Key=artifact_key, Body=artifact)
artifact_version = uploaded.get("VersionId")
model_data_url = f"s3://{bucket}/{artifact_key}"
print("Inference artifact uploaded.")

## 3. Create the model, endpoint configuration, and endpoint

Both the worker and SageMaker allow 1,800 seconds for startup. Use one model
worker to avoid loading duplicate weights. No `VolumeSizeInGB` is set:
SageMaker rejects that setting for `ml.g5.xlarge`.

In [ ]:
sm.create_model(
    ModelName=name,
    ExecutionRoleArn=execution_role,
    PrimaryContainer={
        "Image": inference_image,
        "ModelDataUrl": model_data_url,
        "Environment": {
            "SAGEMAKER_PROGRAM": "inference.py",
            "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/model/code",
            "SAGEMAKER_MODEL_SERVER_WORKERS": "1",
            "SAGEMAKER_MODEL_SERVER_TIMEOUT": "60",
            "TS_DEFAULT_RESPONSE_TIMEOUT": "60",
            "TS_DEFAULT_STARTUP_TIMEOUT": "1800",
            "HF_HOME": "/tmp/huggingface",
            "HF_HUB_DISABLE_TELEMETRY": "1",
            "TOKENIZERS_PARALLELISM": "false",
        },
    },
)
sm.create_endpoint_config(
    EndpointConfigName=name,
    ProductionVariants=[{
        "VariantName": "AllTraffic", "ModelName": name,
        "InitialInstanceCount": 1, "InstanceType": instance_type,
        "InitialVariantWeight": 1.0,
        "ModelDataDownloadTimeoutInSeconds": 900,
        "ContainerStartupHealthCheckTimeoutInSeconds": 1800,
    }],
)
sm.create_endpoint(EndpointName=name, EndpointConfigName=name)
print(f"Endpoint {name} is deploying.")

In [ ]:
sm.get_waiter("endpoint_in_service").wait(
    EndpointName=name, WaiterConfig={"Delay": 30, "MaxAttempts": 120}
)
endpoint = sm.describe_endpoint(EndpointName=name)
assert endpoint["EndpointStatus"] == "InService"
assert endpoint["EndpointConfigName"] == name
assert sum(v["CurrentInstanceCount"] for v in endpoint["ProductionVariants"]) == 1
print("Endpoint is InService with one instance.")

## 4. Verify the runtime and invoke typed questions

The companion adapter accepts the official System One request shape through
SageMaker's IAM-authenticated API. The diagnostics operation reports effective
runtime versions and pinned model identities.

In [ ]:
def invoke(payload):
    response = runtime.invoke_endpoint(
        EndpointName=name, ContentType="application/json",
        Accept="application/json", Body=json.dumps(payload),
    )
    return json.loads(response["Body"].read())

diagnostics = invoke({"operation": "diagnostics"})
assert diagnostics["device"] == "cuda"
assert diagnostics["model_revision"] == "bb282d786bc251fd4e3068de3ada9ddbb38127cd"
assert diagnostics["code_revision"] == "75c9fd32e664954cdc18481434018aa507eee8fb"
assert diagnostics["source_url"] == (
    "https://github.com/strands-labs/strands-decider/archive/"
    "75c9fd32e664954cdc18481434018aa507eee8fb.tar.gz"
)
assert diagnostics["base_revision"] == "b1485b2fa6dfa1287294f269f5fb618e03d52d7c"
assert diagnostics["max_length"] == 4096 and diagnostics["strict_window"]
assert diagnostics["max_batch"] == 4
assert diagnostics["versions"]["torch"].startswith("2.7.1")
assert diagnostics["versions"]["transformers"] == "5.17.0"
assert diagnostics["versions"]["peft"] == "0.21.0"
print(json.dumps(diagnostics, indent=2))

### Tool-call readiness: Noul, Choice, and Score

The user requested weather without supplying a city, but a proposed tool call
guessed Seattle. Ask whether the arguments are grounded, what should happen
next, and how ready the call is. The harness still decides what to execute.

In [ ]:
request = json.loads((example_dir / "example-request.json").read_text())
result = invoke(request)
print(json.dumps(result, indent=2))

In [ ]:
assert result["model"] == diagnostics["model"]
answers = result["answers"]
assert set(answers) == {"arguments_grounded", "next_action", "readiness"}
assert answers["arguments_grounded"]["type"] == "noul"
assert 0 <= answers["arguments_grounded"]["noul"] <= 1
for key, kind in [("next_action", "choice"), ("readiness", "score")]:
    answer = answers[key]
    assert answer["type"] == kind
    assert 0 <= answer["confidence"] <= 1
    probabilities = list(answer["probabilities"].values())
    assert all(math.isfinite(p) and 0 <= p <= 1 for p in probabilities)
    assert abs(sum(probabilities) - 1) <= len(probabilities) * 0.00005 + 1e-6
assert answers["next_action"]["choice"] in request["questions"]["next_action"]["criteria"]
assert 0 <= answers["readiness"]["score"] <= 2
print("All three answer types verified.")

### Basic expected-answer smoke checks

These simple cases check the serving path. They are not a representative
benchmark of decision quality or calibration.

In [ ]:
language = invoke({
    "state": "This is a clear sentence written in English.",
    "questions": {"language": {
        "type": "choice", "instructions": "Which language is this sentence written in?",
        "criteria": {"English": "English", "French": "French", "German": "German"},
    }},
})
assert language["answers"]["language"]["choice"] == "English"

city_question = {"city_supplied": {
    "type": "noul", "instructions": "Did the user explicitly provide a city?",
}}
explicit_city = invoke({
    "state": "The user says: What is the current weather in Paris?",
    "questions": city_question,
})
missing_city = invoke({
    "state": "The user says: What is the current weather?",
    "questions": city_question,
})
assert explicit_city["answers"]["city_supplied"]["noul"] >= 0.5
assert missing_city["answers"]["city_supplied"]["noul"] < 0.5
print("Basic expected-answer checks: 3/3.")

### Repeatability and warm server latency

Three repeats give a smoke measurement, not a production latency estimate.
`latency_ms` excludes SDK/network time.

In [ ]:
repeats = [invoke(request) for _ in range(3)]
assert all(item["answers"] == result["answers"] for item in repeats)
print(f"Median warm server latency: {statistics.median(item['latency_ms'] for item in repeats):.2f} ms")

### Reject oversized state instead of silently truncating it

The oversized request has valid schema. SageMaker wraps the model's HTTP 422
in `ModelError`; its error body may be omitted.

In [ ]:
oversized_request = {
    "state": "weather city location " * 5000,
    "questions": city_question,
}
try:
    invoke(oversized_request)
except runtime.exceptions.ModelError as error:
    assert error.response.get("OriginalStatusCode") == 422, error.response
    print("Oversized state rejected with model HTTP 422.")
else:
    raise AssertionError("The strict context window did not reject oversized state.")

## 5. Cleanup

Delete this example's endpoint and wait before deleting its configuration
and model. Delete its S3 archive and CloudWatch log group; preserve the
existing execution role and shared bucket. Only confirmed missing resources
are ignored. If deployment is still in progress, wait for a terminal state
and rerun cleanup.

Late endpoint log delivery can recreate the CloudWatch log group after its
deletion. Rerun this cell after delivery settles if the group reappears.

In [ ]:
def ignore_missing(operation, **arguments):
    try:
        return operation(**arguments)
    except ClientError as error:
        message = error.response["Error"].get("Message", "")
        error_code = error.response["Error"]["Code"]
        if error_code == "ResourceNotFoundException" or (
            error_code == "ValidationException"
            and ("Could not find" in message or "does not exist" in message)
        ):
            return None
        raise

endpoint_info = ignore_missing(sm.describe_endpoint, EndpointName=name)
if endpoint_info is not None:
    sm.delete_endpoint(EndpointName=name)
    sm.get_waiter("endpoint_deleted").wait(
        EndpointName=name, WaiterConfig={"Delay": 15, "MaxAttempts": 120}
    )
ignore_missing(sm.delete_endpoint_config, EndpointConfigName=name)
ignore_missing(sm.delete_model, ModelName=name)
delete_artifact_args = {"Bucket": bucket, "Key": artifact_key}
if artifact_version is not None:
    delete_artifact_args["VersionId"] = artifact_version
s3.delete_object(**delete_artifact_args)
ignore_missing(logs.delete_log_group, logGroupName=f"/aws/sagemaker/Endpoints/{name}")
archive_path.unlink(missing_ok=True)
assert ignore_missing(sm.describe_endpoint, EndpointName=name) is None
assert ignore_missing(sm.describe_endpoint_config, EndpointConfigName=name) is None
assert ignore_missing(sm.describe_model, ModelName=name) is None
print("Endpoint, configuration, model, S3 archive, and endpoint logs cleaned up.")